# Agent Grass (Phase 5: MQTT Subscriber + Publisher)

This notebook manages binary grass regrowth and publishes grass state each tick.

Phase 5 scope:
- Subscribe: `.../sim/tick`, `.../sim/events`
- Publish: `.../sim/grass/state`

In [1]:
# Cell purpose: import dependencies and connect to MQTT using project config.
from datetime import datetime, timezone
import json
import random
import time

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher

config = load_config()
print(f"Loaded config. Primary MQTT profile: {config.mqtt.host}:{config.mqtt.port}")

connector = MqttConnector(config.mqtt, client_id_suffix="agent-grass-phase5")
connector.connect()
connected = connector.wait_for_connection(timeout=5)
print(f"MQTT connected: {connected}")

publisher = MqttPublisher(connector)
print("STARTUP_OK: agent_grass connected and ready")

Loaded config. Primary MQTT profile: 127.0.0.1:1883
MQTT connected: True
STARTUP_OK: agent_grass connected and ready


In [2]:
# Cell purpose: define topics and initialize binary grass regrowth state.
base_topic = config.mqtt.base_topic.rstrip("/")
tick_topic = f"{base_topic}/sim/tick"
events_topic = f"{base_topic}/sim/events"
grass_state_topic = f"{base_topic}/sim/grass/state"

simulation_cfg = config.simulation
grid_width = simulation_cfg.grid_width if simulation_cfg is not None else 10
grid_height = simulation_cfg.grid_height if simulation_cfg is not None else 10
regrow_ticks = simulation_cfg.grass_regrow_ticks if simulation_cfg is not None else 5
initial_coverage_pct = (
    simulation_cfg.initial_grass_coverage_pct if simulation_cfg is not None else 70
)
seed = (simulation_cfg.seed + 2000) if (simulation_cfg is not None and simulation_cfg.seed is not None) else 2042
rng = random.Random(seed)

total_cells = grid_width * grid_height
grown_cells_target = int((initial_coverage_pct / 100.0) * total_cells)
grown_cells_target = max(0, min(total_cells, grown_cells_target))

# cooldowns[i] = 0 means grass is grown on cell i, >0 means regrowing countdown
cooldowns = [0] * total_cells
initial_not_grown = total_cells - grown_cells_target
if initial_not_grown > 0:
    for idx in rng.sample(range(total_cells), initial_not_grown):
        cooldowns[idx] = regrow_ticks

last_tick = 0
pending_grass_consumed = 0

print("Subscribe topics:")
print(f"- {tick_topic}")
print(f"- {events_topic}")
print("Publish topics:")
print(f"- {grass_state_topic}")

Subscribe topics:
- simulated-city/sim/tick
- simulated-city/sim/events
Publish topics:
- simulated-city/sim/grass/state


In [3]:
# Cell purpose: consume tick/event messages and publish grass state per tick.
def on_message(client, userdata, msg):
    global pending_grass_consumed, last_tick

    try:
        payload = json.loads(msg.payload.decode("utf-8"))
    except Exception:
        print(f"Skipping non-JSON payload on topic {msg.topic}")
        return

    if msg.topic == events_topic:
        consumed = int(payload.get("sheep_ate_grass", 0))
        if consumed > 0:
            pending_grass_consumed += consumed
        return

    if msg.topic == tick_topic:
        tick = int(payload.get("tick", last_tick + 1))
        last_tick = tick

        # Apply sheep consumption by turning grown cells into regrowing cells.
        grown_indices = [index for index, value in enumerate(cooldowns) if value == 0]
        to_consume = min(pending_grass_consumed, len(grown_indices))
        if to_consume > 0:
            for idx in rng.sample(grown_indices, to_consume):
                cooldowns[idx] = regrow_ticks

        pending_grass_consumed = 0

        # Regrow step: decrement cooldown on not-grown cells.
        for index, value in enumerate(cooldowns):
            if value > 0:
                cooldowns[index] = value - 1

        grown_cells = sum(1 for value in cooldowns if value == 0)
        coverage_pct = round((grown_cells / total_cells) * 100.0, 2) if total_cells > 0 else 0.0

        grass_state_payload = {
            "tick": tick,
            "seed": seed,
            "grid_width": grid_width,
            "grid_height": grid_height,
            "grass_grown_cells": grown_cells,
            "grass_coverage_pct": coverage_pct,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        publisher.publish_json(grass_state_topic, json.dumps(grass_state_payload), qos=0)
        print(f"tick={tick} grass_grown={grown_cells}/{total_cells} coverage={coverage_pct}% | published")

connector.client.on_message = on_message
connector.client.subscribe(tick_topic, qos=0)
connector.client.subscribe(events_topic, qos=0)
print("Subscriptions active.")

Subscriptions active.


In [4]:
# Cell purpose: keep this notebook alive to process incoming MQTT messages.
print("Grass agent loop running. Interrupt the cell to stop.")
try:
    while True:
        time.sleep(0.2)
except KeyboardInterrupt:
    print("Stopping grass agent loop.")
finally:
    connector.disconnect()
    print("MQTT disconnected.")

Grass agent loop running. Interrupt the cell to stop.
tick=12 grass_grown=70/100 coverage=70.0% | published


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


tick=13 grass_grown=51/100 coverage=51.0% | published


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=

Stopping grass agent loop.


Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...


MQTT disconnected.
